# AgriEasy Saathi v1 → WebLLM conversion

Downloads the fine-tuned model from HF, converts to q4f16_1 (WebLLM format), uploads the build to a new HF repo. Internet + GPU (verified account).

In [ ]:
# 1) install MLC toolchain (internet available on this account)
%pip install -q --pre -U mlc-ai-nightly mlc-llm-nightly
%pip install -q huggingface_hub
import mlc_llm; print('mlc_llm ready')

In [ ]:
# 2) pull the fine-tuned model from HF
from huggingface_hub import snapshot_download
SRC = snapshot_download("Rishabhthe1/agrieasy-ai-v1")
print("model at", SRC)

In [ ]:
# 3) convert weights to q4f16_1 + generate config (qwen2 chat template)
!python -m mlc_llm convert_weight {SRC} --quantization q4f16_1 -o /kaggle/working/dist
!python -m mlc_llm gen_config {SRC} /kaggle/working/dist --quantization q4f16_1 --conv-template conv_qwen2 --max-seq-len 4096
import os; print(sorted(os.listdir('/kaggle/working/dist')))

In [ ]:
# 4) upload the WebLLM build to Hugging Face
import os
HF_TOKEN = ""   # hf_xxx write token
assert HF_TOKEN, "set HF_TOKEN"
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo("Rishabhthe1/agrieasy-ai-v1-q4f16_1-MLC", repo_type="model", exist_ok=True)
api.upload_folder(folder_path="/kaggle/working/dist", repo_id="Rishabhthe1/agrieasy-ai-v1-q4f16_1-MLC", repo_type="model", commit_message="WebLLM q4f16_1 build of agrieasy-ai-v1")
print("uploaded: huggingface.co/Rishabhthe1/agrieasy-ai-v1-q4f16_1-MLC")